## Ingest Brands Dimension Data into Bronze Layer

### Import required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

   Define schema

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print (bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","shopvista","catalog")
dbutils.widgets.text("data_source", "brands", "data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source= dbutils.widgets.get("data_source")

print (catalog, data_source)

In [0]:
# Define schema for the brands data

brands_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
    
])

In [0]:
# Show the path to the source

base_path = f"s3://shopvista-sv/{data_source}/*.csv"

print(base_path)

In [0]:
#Read data into dataframe and add metadata

df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_path")
)

display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
# write data to bronze table
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")